#### **Step 1: Import Required Libraries**


In [20]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp)
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 22, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [21]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/kafka_eventstream/raw_stream/match_metadata"
df_raw_match_metadata = spark.read.format("parquet").load(raw_transaction_path)

display(df_raw_match_metadata.limit(20))

StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 23, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, efe63801-8371-42a7-80cb-7b5dd8bb09e7)

#### **Step 3: Inspect the live_odds_update dataframe for data quality checks**

In [22]:
# Select specific columns and count nulls
null_counts = df_raw_match_metadata.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
    ['match_id', 'sport', 'league', 'home_team', 'away_team', 'match_start_time', 'status', 'event_date'] 
    ]
)

# Display results
null_counts.show()

StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 24, Finished, Available, Finished)

+--------+-----+------+---------+---------+----------------+------+----------+
|match_id|sport|league|home_team|away_team|match_start_time|status|event_date|
+--------+-----+------+---------+---------+----------------+------+----------+
|       0|    0|     0|        0|        0|               0|     0|         0|
+--------+-----+------+---------+---------+----------------+------+----------+



#### **Step 4: Data Transformation to sport column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels ("Football", "Cricket", "Tennis", "Basketball", "Rugby", "Hockey", "Formual 1").
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid outcomes ("Football", "Cricket", "Tennis", "Basketball", "Rugby", "Hockey", "Formual 1") using a canonical character signature map.
- Returned standardized labels: "Football", "Cricket", "Tennis", "Basketball", "Rugby", "Hockey", "Formual 1".
- Labeled unmatched or null values as "Unknown".

In [23]:
# 1. Pre‑compute canonical signatures
def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid sport values
canonical_values = ["Football", "Cricket", "Tennis", "Basketball", "Rugby", "Hockey", "Formual 1", "Baseball"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)

# 2. UDF to normalize jumbled outcomes
@udf(StringType())
def normalize_outcome(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation
df_cleaned_match_metadata = df_raw_match_metadata.withColumn(
    "sport",
    normalize_outcome(col("sport"))
)

# 4. Drop rows with unrecognized/null sport values
df_cleaned_match_metadata = df_cleaned_match_metadata.filter(col("sport").isNotNull())

# 5. Inspect cleaned sport values
df_cleaned_match_metadata.select("sport").distinct().show(truncate=False)


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 25, Finished, Available, Finished)

+----------+
|sport     |
+----------+
|Tennis    |
|Basketball|
|Baseball  |
|Cricket   |
|Rugby     |
|Hockey    |
|Football  |
|Formual 1 |
+----------+



#### **Step 5: Data Transformation to league column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels ("NBA", "Premier League", "IPL", "ATP Tour", "MLB", "Six Nations", "FIA Formula One").
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid outcomes ("NBA", "Premier League", "IPL", "ATP Tour", "MLB", "Six Nations", "FIA Formula One") using a canonical character signature map.
- Returned standardized labels: "NBA", "Premier League", "IPL", "ATP Tour", "MLB", "Six Nations", "FIA Formula One".
- Labeled unmatched or null values as "Unknown".

In [24]:
# 1. Pre‑compute canonical signatures
def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a lowercase word."""
    return "".join(sorted(word.lower().replace(" ", "")))  # Strip spaces for robustness

# Define the valid league values
canonical_values = ["NBA", "Premier League", "IPL", "ATP Tour", "MLB", "Six Nations", "Formula 1", "NHL"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)

# 2. UDF to normalize jumbled league values
@udf(StringType())
def normalize_league(raw: str) -> str:
    if raw is None:
        return None
    signature = canonical_sig(raw.strip())  # Use same logic as canonical_sig
    return sig_broadcast.value.get(signature)

# 3. Apply transformation
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "league",
    normalize_league(col("league"))
)

# 4. Drop rows with unrecognized/null league values
df_cleaned_match_metadata = df_cleaned_match_metadata.filter(col("league").isNotNull())

# 5. Inspect cleaned league values
df_cleaned_match_metadata.select("league").distinct().show(truncate=False)


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 26, Finished, Available, Finished)

+--------------+
|league        |
+--------------+
|Premier League|
|ATP Tour      |
|IPL           |
|MLB           |
|NHL           |
|Six Nations   |
|NBA           |
+--------------+



#### **Step 6: Data Enrichment — Mapping Sport to League**

Issue:
- The league column was missing.
- A mapping between each sport and its respective league was needed.
- Some sport values might not exist in the defined mapping.

Transformation:
- Created a dictionary mapping known sports to their respective leagues.
- Used a PySpark UDF to assign the correct league based on the sport column.
- Added a new league column to the DataFrame.
- Unrecognized sport values are labeled as "Unknown League".

In [25]:
# Define the mapping dictionary
sport_to_league = {
    "Football": "Premier League",
    "Basketball": "NBA",
    "Cricket": "IPL",
    "Tennis": "ATP Tour",
    "Baseball": "MLB",
    "Hockey": "NHL",
    "Rugby": "Six Nations",
    "Formula 1": "Formula 1"
}

# Create the mapping UDF
def map_league(sport):
    return sport_to_league.get(sport, "Unknown League")

map_league_udf = udf(map_league, StringType())

# Apply the UDF to create a new column 'league'
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "league", map_league_udf(df_cleaned_match_metadata["sport"])
)

# Optional: show result
df_cleaned_match_metadata.show(truncate=False)


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 27, Finished, Available, Finished)

+---------+----------+--------------+----------------+---------------+-------------------+---------+----------+
|match_id |sport     |league        |home_team       |away_team      |match_start_time   |status   |event_date|
+---------+----------+--------------+----------------+---------------+-------------------+---------+----------+
|match_86 |Hockey    |NHL           |South Madison   |Allenchester   |2025-07-05 10:11:19|Live     |2025-07-05|
|match_18 |Baseball  |MLB           |Lake Juan       |Timothyborough |2025-07-05 10:08:19|Finished |2025-07-05|
|match_18 |Hockey    |NHL           |Bryantview      |Patriciabury   |2025-07-05 08:51:19|Fdnieish |2025-07-05|
|match_18 |Baseball  |MLB           |Spencehaven     |Middletonton   |2025-07-05 09:47:19|Scheduled|2025-07-05|
|match_37 |Rugby     |Six Nations   |South Melanie   |Yeseniatown    |2025-07-05 10:09:19|Scheduled|2025-07-05|
|match_100|Tennis    |ATP Tour      |Timothyview     |Yangmouth      |2025-07-05 10:10:19|Finished |2025

#### **Step 7: Data Transformation to status column**
Issue:
- Some values were null, while most were jumbled permutations of valid labels (Finished, Live, Scheduled).
- Inconsistent casing and character arrangements made the values unreliable for analysis.

Transformation:
- Trimmed whitespaces.
- Converted values to lowercase.
- Sorted characters in each value and matched them to valid outcomes (Finished, Live, Scheduled) using a canonical character signature map.
- Returned standardized labels: Finished, Live, Scheduled.
- Labeled unmatched or null values as "Unknown".

In [26]:
# 1. Pre‑compute canonical signatures
def canonical_sig(word: str) -> str:
    """Return alphabetically-sorted characters of a word."""
    return "".join(sorted(word.lower()))

# Define the valid status values
canonical_values = ["Finished", "Live", "Scheduled"]
sig_map = {canonical_sig(w): w for w in canonical_values}

# Broadcast the signature map for performance
sig_broadcast = spark.sparkContext.broadcast(sig_map)

# 2. UDF to normalize jumbled statuses
@udf(StringType())
def normalize_outcome(raw: str) -> str:
    if raw is None:
        return None  # Return None to allow dropping later
    signature = "".join(sorted(raw.lower().strip()))
    return sig_broadcast.value.get(signature)


# 3. Apply transformation
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "status",
    normalize_outcome(col("status"))
)

# 4. Drop rows with unrecognized/null status values
df_cleaned_match_metadata = df_cleaned_match_metadata.filter(col("status").isNotNull())

# 5. Inspect cleaned status values
df_cleaned_match_metadata.select("status").distinct().show(truncate=False)

StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 28, Finished, Available, Finished)

+---------+
|status   |
+---------+
|Finished |
|Scheduled|
|Live     |
+---------+



#### **Step 8: Data Transformation to match_id column**
Issue:
- Some values had inconsistent casing like MATCH_867, match_432, or mixed-case variants.

Transformation:
- Converted all match_id values to lowercase.
- Trimmed any leading or trailing whitespace.

In [27]:
# Standardize `match_id` by converting to lowercase and trimming whitespace
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "match_id",
    lower(trim(col("match_id").cast("string")))
)

# Show sample records after transformation
df_cleaned_match_metadata.select("match_id").show(10, truncate=False)


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 29, Finished, Available, Finished)

+---------+
|match_id |
+---------+
|match_86 |
|match_18 |
|match_18 |
|match_18 |
|match_37 |
|match_100|
|match_100|
|match_93 |
|match_26 |
|match_93 |
+---------+
only showing top 10 rows



#### **Step 10: Data Transformation to match_start_time column**
Issue:
- The column values were in ISO 8601 string format (yyyy-MM-ddTHH:mm:ssZ), not usable for date operations.

Transformation:
- Used to_timestamp() to convert string to proper timestamp format.
- Ensured time zone marker 'Z' was treated as a literal.

In [28]:
# Convert ISO 8601 format to timestamp
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "match_start_time",
    to_timestamp(col("match_start_time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Show sample converted values
df_cleaned_match_metadata.select("match_start_time").show(5, truncate=False)

display(df_cleaned_match_metadata.show())


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 30, Finished, Available, Finished)

+-------------------+
|match_start_time   |
+-------------------+
|2025-07-05 10:11:19|
|2025-07-05 10:08:19|
|2025-07-05 08:51:19|
|2025-07-05 09:47:19|
|2025-07-05 10:09:19|
+-------------------+
only showing top 5 rows

+---------+----------+--------------+----------------+---------------+-------------------+---------+----------+
| match_id|     sport|        league|       home_team|      away_team|   match_start_time|   status|event_date|
+---------+----------+--------------+----------------+---------------+-------------------+---------+----------+
| match_86|    Hockey|           NHL|   South Madison|   Allenchester|2025-07-05 10:11:19|     Live|2025-07-05|
| match_18|  Baseball|           MLB|       Lake Juan| Timothyborough|2025-07-05 10:08:19| Finished|2025-07-05|
| match_18|    Hockey|           NHL|      Bryantview|   Patriciabury|2025-07-05 08:51:19| Finished|2025-07-05|
| match_18|  Baseball|           MLB|     Spencehaven|   Middletonton|2025-07-05 09:47:19|Scheduled|2025-

#### **Step 11: Data Load to Silver Lakehouse Layer**
This step writes the cleaned users data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/kafka_eventstream/cleaned/cleaned_match_metdata.

Prepare Columns for Consistency:
- match_start_time: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [29]:
# Define paths
silver_path = "Files/cdp_silver/kafka_evenstream/cleaned/cleaned_match_metdata"

# Add date column for partitioning (based on event_date if available)
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "event_date", to_date("event_date")  
)

# Add a consistent load timestamp column (once per job run)
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_match_metadata = df_cleaned_match_metadata.withColumn(
    "load_timestamp", lit(load_ts)
)

# Perform MERGE if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target").merge(
        df_cleaned_match_metadata.alias("source"),
        "target.match_id = source.match_id"  # Replace with your unique key
    ).whenNotMatchedInsertAll().execute()

else:
    # First write: create partitioned Delta table
    df_cleaned_match_metadata.write.format("delta") \
        .mode("append") \
        .partitionBy("event_date") \
        .save(silver_path)


StatementMeta(, f9c307ed-2a50-4d60-a63b-18a28439faac, 31, Finished, Available, Finished)